# BWE Stufe 2 — GAN-Training (Kaggle/GPU)

**Auf Kaggle ausführen.** Setup wie `kaggle_train_regression.ipynb`:

1. **Accelerator → GPU** (T4 ×2 oder P100); braucht ggf. Telefon-Verifizierung (schaltet GPU **und** Internet frei).
2. **Internet → On** (sonst scheitert `git clone`).
3. **Add Data → MUSDB18-HQ** einbinden und in Zelle 2 `BWE_DATA_ROOT` darauf zeigen lassen (Ordner mit `train/` und `test/`; ein evtl. `val/` wird ignoriert).
4. **Add Data → Phase-C-Run** (Output-Dataset mit `reg_full/generator.weights.h5`) für den **Warm-Start** — oder zuerst `kaggle_train_regression.ipynb` in dieser Session laufen lassen.

Das GAN feintunt den **warm-gestarteten** Generator adversarial gegen einen PatchGAN-Diskriminator (Spectral Norm). `g_loss` ist **kein** Qualitätsmaß — Modellwahl per `val_lsd_hf` + Reinhören (`03_gan.ipynb`).

**Workflow wie bei der Regression:** **eine** Variable `FULL_TRAINING` (Zelle 1) steuert alles — `False` = kleines Subset im *Edit-Modus*, `True` = Volllauf mit 32-kHz-Cache im *Commit-Modus* (*Save & Run All*). Nichts auskommentieren; die jeweils inaktive Zelle überspringt sich selbst.

In [ ]:
FULL_TRAINING = True  # True: 32kHz-Cache + Voll-GAN (Commit-Modus). False: kein Cache + Subset (Edit-Modus).

# GPU prüfen
import tensorflow as tf
print("TF", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))
print(f"FULL_TRAINING = {FULL_TRAINING}")

## 1. Code holen & installieren

TF ist auf Kaggle vorinstalliert → `--no-deps`, damit die GPU-Build nicht ersetzt wird; nur die Audio-Pakete nachinstallieren.

In [ ]:
REPO = "https://github.com/DanyelRychter/bwe_dnn.git"
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    REPO = REPO.replace("https://", f"https://{tok}@")
except Exception:
    pass  # öffentliches Repo

import subprocess
subprocess.run(["git", "clone", "--depth", "1", REPO, "/kaggle/working/bwe_dnn"], check=True)
%pip install -q -e /kaggle/working/bwe_dnn --no-deps
%pip install -q librosa soundfile soxr

## 2. Pfade setzen (VOR dem bwe-Import!)

`BWE_DATA_ROOT` = Ordner mit `train/` und `test/`. Ein evtl. `val/` wird ignoriert (kanonischer Split via Track-Namen).

In [ ]:
import os
# >>> An die tatsächliche Dataset-Struktur anpassen: <<<
os.environ["BWE_DATA_ROOT"] = "/kaggle/input/datasets/quanglvitlm/musdb18-hq"     # muss train/ und test/ enthalten
os.environ["BWE_CKPT_ROOT"] = "/kaggle/working/bwe_runs"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import sys; sys.path.insert(0, "/kaggle/working/bwe_dnn")
from bwe import config as cfg
print(cfg.summary())
print(f"\nGAN: λ_adv={cfg.LAMBDA_ADV}  λ_fm={cfg.LAMBDA_FM}  lr={cfg.GAN_LR}  beta_1={cfg.GAN_BETA_1}  "
      f"batch={cfg.GAN_BATCH_SIZE}  epochs={cfg.GAN_EPOCHS}  D-Vorlauf={cfg.GAN_D_WARMUP_STEPS} Steps")

In [ ]:
# Datensatz-Check: erwartete 86/14/50 (bzw. tatsächliche Zahlen)
from bwe.data import splits as SP
for s in SP.SPLIT_NAMES:
    print(f"{s:6s}: {len(SP.get_split(s)):3d} Tracks")

## 2b. 32-kHz-Cache (empfohlen fürs Volltraining)

Dieselbe Logik wie im Regressions-Notebook: `use_resample_cache` (aus `bwe/data/cache.py`)
resampelt die 4 Stems **aller** Tracks **einmal** nach `/kaggle/working/musdb18hq_32k` und biegt
`cfg.DATA_ROOT` darauf → im Training gilt `sr == cfg.SR`, der On-the-fly-Resample-Zweig entfällt
(GPU-Bound statt I/O-Bound). Stereo bleibt erhalten (Kanal-Augmentation), `mixture.wav` wird nicht
gecacht, resumebar.

**Gesteuert durch `FULL_TRAINING`**: nur bei `True` wird gecacht; bei `False` (Subset) bleibt
`cfg.DATA_ROOT` auf den 44,1-kHz-Daten (bei 20 Tracks schnell genug). **Commit-Modus:**
`/kaggle/working` ist je Lauf frisch → der Cache wird jedes Mal neu gebaut (~13 min/150 Tracks).

In [ ]:
if FULL_TRAINING:
    from bwe.data.cache import use_resample_cache

    use_resample_cache(os.environ["BWE_DATA_ROOT"], "/kaggle/working/musdb18hq_32k")
    for s in SP.SPLIT_NAMES:                        # Gegencheck aus dem Cache: erwartet 86/14/50
        print(f"  {s:6s}: {len(SP.get_split(s)):3d} Tracks")

## 3. Warm-Start-Gewichte (Phase C)

Sucht `generator.weights.h5` zuerst in eingebundenen Input-Datasets (bevorzugt `reg_full`), dann
unter `BWE_CKPT_ROOT` (falls die Regression in derselben Session lief). Ohne Warm-Start würde das
GAN from-scratch starten — möglich, aber instabiler und nicht die geplante Curriculum-Logik.

In [ ]:
import glob
pats = ["/kaggle/input/**/reg_full/generator.weights.h5",
        "/kaggle/input/**/reg*/generator.weights.h5",
        "/kaggle/input/**/generator.weights.h5",
        os.path.join(os.environ["BWE_CKPT_ROOT"], "**", "generator.weights.h5")]
cands, seen = [], set()
for p in pats:
    for c in sorted(glob.glob(p, recursive=True)):
        if c not in seen:
            seen.add(c); cands.append(c)
WARM_START = cands[0] if cands else None
print("Warm-Start:", WARM_START or "KEINER gefunden — GAN startet from-scratch (nicht empfohlen).")

## 4. Subset-GAN (Edit-Modus, `FULL_TRAINING = False`)

Wenige Tracks → zuerst prüfen, dass das Training **nicht divergiert** (d_loss/g_loss pendeln, kein
Kollaps) und das HF sichtbar schärfer wird als bei der Regression. Bei Instabilität: `λ_adv` senken /
`λ_fm` erhöhen (in `cfg` oder per CLI). Überspringt sich bei `FULL_TRAINING = True` selbst.

In [ ]:
if not FULL_TRAINING:
    from bwe.train.gan import train
    model, hist = train(run="gan_subset", mode="subset", warm_start=WARM_START,
                        batch_size=cfg.GAN_BATCH_SIZE, epochs=30, limit=20)

## 5. Voll-GAN (Cold-Start oder Resume)

Eine Zelle, drei Fälle — funktioniert im **Commit-Modus** out-of-the-box. Gesteuert über `CKPT_SRC`
(GAN-Resume) plus dem `WARM_START` aus Schritt 3 (Phase-C-Generator):

- **`CKPT_SRC = None`** → **Cold-Start**: Generator aus `WARM_START`, Diskriminator-Vorlauf, dann adversarial.
- **`CKPT_SRC` zeigt auf ein Dataset mit `ckpt/`** → **exaktes Resume** (G, D, **beide** Optimizer,
  Epoche via eigenem `tf.train.Checkpoint`; `WARM_START` wird ignoriert, der Zustand kommt aus `ckpt/`).
- **`CKPT_SRC` nur mit `(best_)generator.weights.h5`** → **Generator-Warm-Start** daraus.

### Resume im Commit-Modus (exakt, inkl. `ckpt/`)
Kaggle sichert bei einem Commit `/kaggle/working` automatisch als **Output der Version** — inklusive
`bwe_runs/gan_full/ckpt/`. Damit klappt exaktes Resume **ohne** einzelne Zellen neu auszuführen:
1. **Add Input → Your Work** → Output der vorigen Commit-Version anhängen.
2. `CKPT_SRC` auf den Pfad im *Input*-Panel setzen (`/kaggle/input/<slug>`, ggf. mit Unterordner `bwe_runs/gan_full`).
3. Erneut committen → liegt `ckpt/` darin, läuft es **exakt an der letzten Epoche** weiter (D-Vorlauf
   entfällt); sonst Generator-Warm-Start.

> Für ein Resume **über `cfg.GAN_EPOCHS` hinaus** `epochs` erhöhen. Der Abschlussgrund steht
> explizit im Log (`>>> ... <<<`), sodass ein sauberer Stopp von einem Abbruch unterscheidbar ist.

In [ ]:
# Voll-GAN -- Cold-Start ODER Resume, je nach CKPT_SRC (siehe Markdown oben).
#   CKPT_SRC = None                          -> Cold-Start (Generator aus WARM_START)
#   CKPT_SRC mit .../ckpt/                   -> exaktes Resume (G, D, beide Optimizer, Epoche)
#   CKPT_SRC nur mit generator.weights.h5    -> Generator-Warm-Start
if FULL_TRAINING:
    from bwe.train.gan import train_resumable

    # >>> Zum Fortsetzen: angehaengtes Checkpoint-Dataset eintragen, sonst None lassen. <<<
    CKPT_SRC = None   # z.B. "/kaggle/input/bwe-gan-full-ckpt/bwe_runs/gan_full"

    model, hist = train_resumable(run="gan_full", mode="full", warm_start=WARM_START,
                                  ckpt_src=CKPT_SRC, batch_size=cfg.GAN_BATCH_SIZE,
                                  epochs=cfg.GAN_EPOCHS)

## 6. Lernkurven

Gewichte unter `bwe_runs/gan_full/` (`generator.weights.h5` = letzter Stand,
`best_generator.weights.h5` = bestes Val-LSD-HF, `discriminator.weights.h5`). **Wichtig:**
`g_loss`/`d_loss` sind kein Qualitätsmaß — sie sollen nur *pendeln*, nicht kollabieren. Qualität
steht in `val_lsd_hf`.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

run = "gan_full" if FULL_TRAINING else "gan_subset"
log = pd.read_csv(os.path.join(os.environ["BWE_CKPT_ROOT"], run, "log.csv"))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(log["epoch"], log["d_loss"], label="d_loss")
ax[0].plot(log["epoch"], log["g_loss"], label="g_loss")
ax[0].plot(log["epoch"], log["recon"], label="recon", ls="--")
ax[0].set_title("GAN-Losses (kein Qualitätsmaß!)"); ax[0].set_xlabel("Epoche"); ax[0].legend()
ax[1].plot(log["epoch"], log["lsd_hf"], label="train")
ax[1].plot(log["epoch"], log["val_lsd_hf"], label="val")
ax[1].set_title("LSD-HF [dB] (Qualitätsmaß)"); ax[1].set_xlabel("Epoche"); ax[1].legend()
plt.tight_layout(); plt.show()